In [4]:
!pip install opencv-python
!pip install tensorflow.keras

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.preprocessing.image import ImageDataGenerator

# Define the number of classes (ASL signs) in the dataset.
# For example, for 26 signs for the alphabet, set it to 26.
num_asl_signs = 26

# Initialising the CNN for multi-class
classifier = Sequential()

# Step 1 - Convolution
classifier.add(Conv2D(32, (3, 3), input_shape = (64, 64, 3), activation = 'relu'))

# Step 2 - Pooling
classifier.add(MaxPooling2D(pool_size = (2, 2)))

# Adding a second convolutional layer
classifier.add(Conv2D(32, (3, 3), activation = 'relu'))
classifier.add(MaxPooling2D(pool_size = (2, 2)))

# Step 3 - Flattening
classifier.add(Flatten())

# Step 4 - Full connection (MODIFIED for multi-class)
classifier.add(Dense(units = 128, activation = 'relu'))
# The final Dense layer now has 'num_asl_signs' units and 'softmax' activation
# for outputting a probability distribution over all classes.
classifier.add(Dense(units = num_asl_signs, activation = 'softmax'))

# Compiling the CNN
# For multi-class classification, 'categorical_crossentropy' is the appropriate loss function.
classifier.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

# Fitting the CNN to the images

train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)

test_datagen = ImageDataGenerator(rescale = 1./255)

training_set = train_datagen.flow_from_directory(
    './your_train_data_path', # add training dataset path here
    target_size = (64, 64),
    batch_size = 32,
    class_mode = 'categorical')

test_set = test_datagen.flow_from_directory(
    './your_test_data_path', # add testing dataset path here
    target_size = (64, 64),
    batch_size = 32,
    class_mode = 'categorical')

classifier.fit_generator(training_set,
                         steps_per_epoch = len(training_set),
                         epochs = 15,
                         validation_data = test_set,
                         validation_steps = len(test_set))

D:\anaconda\lib\site-packages\h5py\__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


Found 30 images belonging to 2 classes.
Found 10 images belonging to 2 classes.
Epoch 1/15
15/15 [==============================] - 22s 1s/step - loss: 0.7333 - acc: 0.5533 - val_loss: 0.6863 - val_acc: 0.7000
Epoch 2/15
15/15 [==============================] - 17s 1s/step - loss: 0.6498 - acc: 0.7200 - val_loss: 0.6383 - val_acc: 0.8000
Epoch 3/15
15/15 [==============================] - 18s 1s/step - loss: 0.4747 - acc: 0.8933 - val_loss: 0.8051 - val_acc: 0.8000
Epoch 4/15
15/15 [==============================] - 18s 1s/step - loss: 0.2605 - acc: 0.9289 - val_loss: 1.1489 - val_acc: 0.8000
Epoch 5/15
15/15 [==============================] - 17s 1s/step - loss: 0.1450 - acc: 0.9689 - val_loss: 1.4560 - val_acc: 0.8000
Epoch 6/15
15/15 [==============================] - 17s 1s/step - loss: 0.0632 - acc: 0.9978 - val_loss: 1.6945 - val_acc: 0.8000
Epoch 7/15
15/15 [==============================] - 17s 1s/step - loss: 0.0347 - acc: 0.9978 - val_loss: 2.0364 - val_acc: 0.7000
Epoch 8/15

## Save the Trained Model

To make your project standalone and avoid retraining the model every time, it's crucial to save the trained model and the class-to-label mapping. This block saves your `classifier` and the `training_set.class_indices`.

In [ ]:
import os
import json

# Define the directory to save model assets
model_save_dir = 'asl_model_assets'
os.makedirs(model_save_dir, exist_ok=True)

# Save the model architecture and weights
model_path = os.path.join(model_save_dir, 'asl_classifier_model.h5')
classifier.save(model_path)
print(f"Model saved to: {model_path}")

# Save the class indices mapping
class_indices_path = os.path.join(model_save_dir, 'class_indices.json')
with open(class_indices_path, 'w') as f:
    json.dump(training_set.class_indices, f)
print(f"Class indices saved to: {class_indices_path}")


In [ ]:
# to test single image
import numpy as np
from keras.preprocessing import image

single_image_path = './single_prediction/my_asl_sign_image.jpg' # path to test image
test_image = image.load_img(single_image_path, target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)

# Predict probabilities for each class
predictions = classifier.predict(test_image)

# Get the index of the class with the highest probability
predicted_class_index = np.argmax(predictions[0])

# To get the actual class label from the index, class indices mapping is used.
# Reverse this mapping to get the class name from its predicted index.
label_map = training_set.class_indices
inverse_label_map = dict((v, k) for k, v in label_map.items())

prediction_label = inverse_label_map[predicted_class_index]

print(f"The predicted ASL sign is: {prediction_label}")

a


## Live Video ASL Prediction

This code block will attempt to capture video from your webcam (usually `index=0`). For each frame, it will:

1.  **Read a frame:** Capture a single frame from the camera.
2.  **Preprocess:** Resize the frame to 64x64 pixels, convert it to an array, normalize pixel values, and expand dimensions to match the model's expected input.
3.  **Predict:** Use the trained `classifier` model to predict the ASL sign.
4.  **Display:** Overlay the predicted sign on the video frame.

Press the 'q' key to quit the video stream.

**Note:** If you are running this in Google Colab,You might need to use a library like `ipywidgets` or `google.colab.patches` for interactive webcam streaming. The code below is a standard OpenCV implementation that work best in a local environment or a Colab runtime with direct webcam access.

In [ ]:
import cv2
import numpy as np
from keras.preprocessing import image

# Open the default camera, index `0` refers to the default webcam.
# for multiple cameras, try other indices (1, 2, etc.)
cap = cv2.VideoCapture(0)

# Check if the camera opened successfully
if not cap.isOpened():
    print("Error: Could not open video stream. Please check your webcam connection or permissions.")
else:
    print("Webcam opened successfully. Press 'q' to quit.")

    # Get the class labels in the correct order
    label_map = training_set.class_indices
    inverse_label_map = dict((v, k) for k, v in label_map.items())

    while(True):
        # Capture frame-by-frame
        ret, frame = cap.read()

        if not ret:
            print("Error: Failed to grab frame.")
            break

        # Preprocess the frame for the model
        # Resize to 64x64
        resized_frame = cv2.resize(frame, (64, 64))
        test_image = image.img_to_array(resized_frame)
        test_image = np.expand_dims(test_image, axis = 0)
        # Normalize pixel values
        test_image = test_image / 255.0

        # Predict probabilities for each class
        predictions = classifier.predict(test_image)

        # Get the index of the class with the highest probability
        predicted_class_index = np.argmax(predictions[0])
        prediction_label = inverse_label_map[predicted_class_index]

        # Display the prediction on the frame
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(frame, f"Prediction: {prediction_label}", (10, 30), font, 1, (0, 255, 0), 2, cv2.LINE_AA)

        # Display the resulting frame
        cv2.imshow('ASL Live Prediction', frame)

        # Press 'q' on keyboard to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # release the capture
    cap.release()
    cv2.destroyAllWindows()


## Standalone ASL Live Prediction Application

This section provides the code for a standalone Python application. You can copy this code into a `.py` file (e.g., `predict_asl.py`) and run it from your local terminal. It includes:

*   **Argument Parsing:** Using `argparse` for model path, class indices path, and webcam index.
*   **Model Loading:** Loading the saved Keras model.
*   **Live Video Processing:** Using OpenCV to capture video, preprocess frames, and display predictions.
*   **Error Handling:** Basic checks for camera availability and model loading.

**To run this locally:**

1.  **Download the model assets:** After running the above cell, download the `asl_model_assets` folder to your local machine.
2.  **Create `predict_asl.py`:** Copy the Python code from the next cell into a file named `predict_asl.py`.
3.  **Install dependencies:** Ensure you have `tensorflow`, `keras`, `opencv-python`, and `numpy` installed in your local environment (`pip install tensorflow keras opencv-python numpy`).
4.  **Run from terminal:** Navigate to the directory containing `predict_asl.py` and the `asl_model_assets` folder, then run:
    ```bash
    python predict_asl.py --model_path asl_model_assets/asl_classifier_model.h5 --class_indices_path asl_model_assets/class_indices.json
    ```
    You can also specify a different `--webcam_index` if needed.

In [ ]:
# -----------------------------------------------------------------------------
# This is a standalone script designed to be run in a local Python environment
# Copy this code into a file like 'predict_asl.py' and run it from your terminal
# -----------------------------------------------------------------------------

import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import argparse
import json
import os

def main():
    parser = argparse.ArgumentParser(description="ASL Sign Language Live Video Predictor")
    parser.add_argument('--model_path', type=str, default='asl_model_assets/asl_classifier_model.h5',
                        help='Path to the saved Keras model (.h5 file)')
    parser.add_argument('--class_indices_path', type=str, default='asl_model_assets/class_indices.json',
                        help='Path to the saved class_indices.json file')
    parser.add_argument('--webcam_index', type=int, default=0,
                        help='Index of the webcam to use (e.g., 0 for default)')
    args = parser.parse_args()

    # --- 1. Load the Model and Class Indices ---
    try:
        classifier = load_model(args.model_path)
        print(f"Model loaded from: {args.model_path}")
    except Exception as e:
        print(f"Error loading model from {args.model_path}: {e}")
        print("Please ensure the model path is correct and the model file exists.")
        return

    try:
        with open(args.class_indices_path, 'r') as f:
            label_map = json.load(f)
        inverse_label_map = dict((v, k) for k, v in label_map.items())
        print(f"Class indices loaded from: {args.class_indices_path}")
        print(f"Detected classes: {list(label_map.keys())}")
    except Exception as e:
        print(f"Error loading class indices from {args.class_indices_path}: {e}")
        print("Please ensure the class_indices.json file exists and is valid.")
        return

    # --- 2. Initialize Video Capture ---
    cap = cv2.VideoCapture(args.webcam_index)

    if not cap.isOpened():
        print(f"Error: Could not open video stream using webcam index {args.webcam_index}.")
        print("Please check your webcam connection, permissions, or try a different index.")
        return
    else:
        print("Webcam opened successfully. Press 'q' to quit.")

    # --- 3. Live Prediction Loop ---
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame. Exiting...")
            break

        # Preprocess the frame for the model
        # The model was trained on 64x64 images
        resized_frame = cv2.resize(frame, (64, 64))
        processed_image = image.img_to_array(resized_frame)
        processed_image = np.expand_dims(processed_image, axis=0) # Add batch dimension
        processed_image = processed_image / 255.0 # Normalize pixel values to [0, 1]

        # Make prediction
        predictions = classifier.predict(processed_image)
        predicted_class_index = np.argmax(predictions[0])
        prediction_label = inverse_label_map.get(predicted_class_index, 'Unknown') # Handle unknown index

        # Display prediction on frame
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(frame, f"Prediction: {prediction_label}", (10, 30), font, 1, (0, 255, 0), 2, cv2.LINE_AA)

        # Display the resulting frame
        cv2.imshow('ASL Live Prediction', frame)

        # Exit loop on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # --- 4. Release Resources ---
    cap.release()
    cv2.destroyAllWindows()

if __name__ == '__main__':
    main()
